# Construcción de la dimensión de clientes — `silver.dim_cliente`

## Objetivo del notebook

Este notebook construye la primera tabla de la capa Silver: una versión limpia y enriquecida de la dimensión de clientes a partir de los datos crudos almacenados en `bronze.dim_cliente`.

La capa Silver representa el primer nivel de transformación analítica: los datos ya son comparables, los tipos están correctamente casteados, los identificadores son consistentes y se han añadido los campos derivados que el análisis posterior requerirá.

### Transformaciones que se aplicarán

A partir de los hallazgos de los notebooks de exploración (`01_exploracion_general` y `02_exploracion_mosaic`), las decisiones metodológicas a aplicar son:

1. Garantizar el tipo VARCHAR del `id_cliente` para coherencia con las tablas de hechos en JOINs posteriores.
2. Limpiar campos de texto (TRIM en nombre, dirección, localidad) para evitar problemas con espacios en blanco.
3. Normalizar el código postal a cinco dígitos mediante padding (`LPAD`), permitiendo el JOIN posterior con la tabla MOSAIC.
4. Crear el flag `es_cliente_espanol` aplicando una validación cruzada entre el código postal y el código de provincia (los dos primeros dígitos del CP español deben coincidir con el código de provincia).
5. Crear el campo `tipo_mercado` (`NACIONAL` o `INTERNACIONAL`) para diferenciar los dos grandes bloques de la cartera de Selmark.
6. Conservar el CP original en una columna paralela para auditoría y trazabilidad.

### Resultado esperado

Una tabla `silver.dim_cliente` que conserva íntegramente el volumen de la capa Bronze (3.492 registros) e incorpora los campos derivados y normalizados que se requieren para su uso como dimensión maestra de cliente en las capas analíticas posteriores.

## 1. Configuración del entorno

Se establece la conexión con la base DuckDB en **modo escritura** (`read_only=False`). Es el primer notebook que requiere permisos de escritura, ya que va a crear una nueva tabla en el esquema `silver`.

In [33]:
import duckdb
import pandas as pd
from pathlib import Path

RUTA_PROYECTO = Path("..").resolve()
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

# Conexión en modo escritura
con = duckdb.connect(str(RUTA_DUCKDB), read_only=False)
print(f"Conexión establecida con: {RUTA_DUCKDB}")
print(f"Modo: escritura")

Conexión establecida con: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb
Modo: escritura


## 2. Inspección previa de la tabla origen

Antes de transformar, se confirma el estado actual de `bronze.dim_cliente`: número de filas, esquema y muestra de los datos. Esta verificación es importante por si la base se hubiera modificado entre sesiones.

In [34]:
# Conteo y esquema
print(f"Filas en bronze.dim_cliente: {con.execute('SELECT COUNT(*) FROM bronze.dim_cliente').fetchone()[0]:,}\n")

print("Esquema:")
esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'bronze' AND table_name = 'dim_cliente'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema.to_string(index=False))

print("\nMuestra de 5 filas:")
muestra = con.execute("SELECT * FROM bronze.dim_cliente LIMIT 5").fetchdf()
print(muestra.to_string(index=False))

Filas en bronze.dim_cliente: 3,492

Esquema:
             column_name data_type
              id_cliente   VARCHAR
          nombre_cliente   VARCHAR
nombre_comercial_cliente   VARCHAR
       direccion_cliente   VARCHAR
       localidad_cliente   VARCHAR
   codigo_postal_cliente   VARCHAR
codigo_provincia_cliente   VARCHAR

Muestra de 5 filas:
id_cliente          nombre_cliente nombre_comercial_cliente                   direccion_cliente localidad_cliente codigo_postal_cliente codigo_provincia_cliente
         1 PEREZ RODRIGUEZ, AMADOR                      NaN      CL/ CELSO E. FERREIRO - 7 8º A              VIGO                 36203                      036
      1010    PERFECT FIT LINGERIE                      NaN                  16 COURT STREET S.       THONDER BAY                P7B2W3                      CAN
      1024        PAULA-COM S.R.L.        Litvinov Alexandr                ALBA IULIA, 2, AP.12          CHISINAU                MD2064                       MD
      1037

## 3. Construcción de `silver.dim_cliente`

Se ejecuta una sentencia `CREATE OR REPLACE TABLE` que aplica todas las transformaciones en una sola operación. Las claves del proceso son:

- `id_cliente` confirmado como VARCHAR mediante casteo explícito.
- TRIM sobre todos los campos de texto.
- Normalización del CP: si el campo tiene entre 1 y 5 caracteres y son todos dígitos, se aplica padding a 5. En caso contrario se conserva sin transformar.
- Flag `es_cliente_espanol`: validación cruzada estricta entre CP y código de provincia.
- Campo `tipo_mercado`: `NACIONAL` o `INTERNACIONAL` derivado del flag.
- Conservación del CP original en `codigo_postal_original`.

In [35]:
con.execute("""
    CREATE OR REPLACE TABLE silver.dim_cliente AS
    SELECT
        CAST(id_cliente AS VARCHAR) AS id_cliente,
        TRIM(nombre_cliente) AS nombre_cliente,
        TRIM(nombre_comercial_cliente) AS nombre_comercial_cliente,
        TRIM(direccion_cliente) AS direccion_cliente,
        TRIM(localidad_cliente) AS localidad_cliente,
        
        -- CP original para auditoría
        TRIM(codigo_postal_cliente) AS codigo_postal_original,
        
        -- CP normalizado: padding a 5 dígitos solo si es numérico válido
        CASE
            WHEN TRIM(codigo_postal_cliente) IS NULL THEN NULL
            WHEN LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
            THEN LPAD(TRIM(codigo_postal_cliente), 5, '0')
            ELSE TRIM(codigo_postal_cliente)
        END AS codigo_postal_norm,
        
        -- Provincia: normalizar texto 'null' a NULL real
        CASE
            WHEN LOWER(TRIM(codigo_provincia_cliente)) IN ('null', 'nan', '') THEN NULL
            ELSE TRIM(codigo_provincia_cliente)
        END AS codigo_provincia_cliente,
        
        -- Flag de cliente español: 4 reglas combinadas con OR
        CASE
            -- REGLA A: provincia 2 dígitos válida que coincide con CP
            WHEN TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRIM(codigo_provincia_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_provincia_cliente)) = 2
                 AND REGEXP_MATCHES(TRIM(codigo_provincia_cliente), '^[0-9]{2}$')
                 AND TRY_CAST(TRIM(codigo_provincia_cliente) AS INTEGER) BETWEEN 1 AND 52
                 AND SUBSTR(LPAD(TRIM(codigo_postal_cliente), 5, '0'), 1, 2) = TRIM(codigo_provincia_cliente)
            THEN TRUE
            
            -- REGLA B: provincia '0XX' donde los 2 últimos coinciden con CP
            WHEN TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRIM(codigo_provincia_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_provincia_cliente)) = 3
                 AND REGEXP_MATCHES(TRIM(codigo_provincia_cliente), '^0[0-9]{2}$')
                 AND TRY_CAST(SUBSTR(TRIM(codigo_provincia_cliente), 2, 2) AS INTEGER) BETWEEN 1 AND 52
                 AND SUBSTR(LPAD(TRIM(codigo_postal_cliente), 5, '0'), 1, 2) = SUBSTR(TRIM(codigo_provincia_cliente), 2, 2)
            THEN TRUE
            
            -- REGLA C: provincia no informativa (NULL real o texto 'null') + CP español válido
            WHEN (
                    codigo_provincia_cliente IS NULL
                    OR LOWER(TRIM(codigo_provincia_cliente)) IN ('null', 'nan', '')
                 )
                 AND TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) = 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRY_CAST(SUBSTR(TRIM(codigo_postal_cliente), 1, 2) AS INTEGER) BETWEEN 1 AND 52
            THEN TRUE
            
            ELSE FALSE
        END AS es_cliente_espanol,
        
        -- Tipo de mercado: NACIONAL si cumple alguna regla, INTERNACIONAL en caso contrario
        CASE
            -- REGLA A
            WHEN TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRIM(codigo_provincia_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_provincia_cliente)) = 2
                 AND REGEXP_MATCHES(TRIM(codigo_provincia_cliente), '^[0-9]{2}$')
                 AND TRY_CAST(TRIM(codigo_provincia_cliente) AS INTEGER) BETWEEN 1 AND 52
                 AND SUBSTR(LPAD(TRIM(codigo_postal_cliente), 5, '0'), 1, 2) = TRIM(codigo_provincia_cliente)
            THEN 'NACIONAL'
            
            -- REGLA B
            WHEN TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRIM(codigo_provincia_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_provincia_cliente)) = 3
                 AND REGEXP_MATCHES(TRIM(codigo_provincia_cliente), '^0[0-9]{2}$')
                 AND TRY_CAST(SUBSTR(TRIM(codigo_provincia_cliente), 2, 2) AS INTEGER) BETWEEN 1 AND 52
                 AND SUBSTR(LPAD(TRIM(codigo_postal_cliente), 5, '0'), 1, 2) = SUBSTR(TRIM(codigo_provincia_cliente), 2, 2)
            THEN 'NACIONAL'
            
            -- REGLA C
            WHEN (
                    codigo_provincia_cliente IS NULL
                    OR LOWER(TRIM(codigo_provincia_cliente)) IN ('null', 'nan', '')
                 )
                 AND TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) = 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRY_CAST(SUBSTR(TRIM(codigo_postal_cliente), 1, 2) AS INTEGER) BETWEEN 1 AND 52
            THEN 'NACIONAL'
            
            ELSE 'INTERNACIONAL'
        END AS tipo_mercado
        
    FROM bronze.dim_cliente
""")

print("✅ Tabla silver.dim_cliente recreada con las 3 reglas de clasificación")

✅ Tabla silver.dim_cliente recreada con las 3 reglas de clasificación


## 4. Validación de la tabla generada

Se verifica que el resultado es coherente:

1. Que el número de filas es exactamente igual al original.
2. Que la clave primaria sigue siendo única.
3. Que los tipos de datos son los esperados.
4. Que las columnas nuevas tienen distribuciones razonables.

In [36]:
print("VALIDACIÓN DE INTEGRIDAD\n")

filas_bronze = con.execute("SELECT COUNT(*) FROM bronze.dim_cliente").fetchone()[0]
filas_silver = con.execute("SELECT COUNT(*) FROM silver.dim_cliente").fetchone()[0]
ids_unicos = con.execute("SELECT COUNT(DISTINCT id_cliente) FROM silver.dim_cliente").fetchone()[0]

print(f"Filas en bronze: {filas_bronze:,}")
print(f"Filas en silver: {filas_silver:,}")
print(f"id_cliente únicos: {ids_unicos:,}")
print(f"\nIntegridad de filas:  {'OK' if filas_bronze == filas_silver else 'ERROR'}")
print(f"Clave única:          {'OK' if filas_silver == ids_unicos else 'ERROR'}")

VALIDACIÓN DE INTEGRIDAD

Filas en bronze: 3,492
Filas en silver: 3,492
id_cliente únicos: 3,492

Integridad de filas:  OK
Clave única:          OK


In [37]:
print("ESQUEMA DE silver.dim_cliente\n")

esquema = con.execute("""
    SELECT column_name AS columna, data_type AS tipo
    FROM information_schema.columns
    WHERE table_schema = 'silver' AND table_name = 'dim_cliente'
    ORDER BY ordinal_position
""").fetchdf()

print(esquema.to_string(index=False))

ESQUEMA DE silver.dim_cliente

                 columna    tipo
              id_cliente VARCHAR
          nombre_cliente VARCHAR
nombre_comercial_cliente VARCHAR
       direccion_cliente VARCHAR
       localidad_cliente VARCHAR
  codigo_postal_original VARCHAR
      codigo_postal_norm VARCHAR
codigo_provincia_cliente VARCHAR
      es_cliente_espanol BOOLEAN
            tipo_mercado VARCHAR


In [38]:
print("DISTRIBUCIÓN DEL FLAG es_cliente_espanol\n")

dist_flag = con.execute("""
    SELECT 
        es_cliente_espanol,
        COUNT(*) AS num_clientes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM silver.dim_cliente
    GROUP BY es_cliente_espanol
    ORDER BY es_cliente_espanol DESC
""").fetchdf()

print(dist_flag.to_string(index=False))

DISTRIBUCIÓN DEL FLAG es_cliente_espanol

 es_cliente_espanol  num_clientes  porcentaje
               True          2093       59.94
              False          1399       40.06


In [39]:
print("DISTRIBUCIÓN DE tipo_mercado\n")

dist_mercado = con.execute("""
    SELECT 
        tipo_mercado,
        COUNT(*) AS num_clientes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM silver.dim_cliente
    GROUP BY tipo_mercado
    ORDER BY num_clientes DESC
""").fetchdf()

print(dist_mercado.to_string(index=False))

DISTRIBUCIÓN DE tipo_mercado

 tipo_mercado  num_clientes  porcentaje
     NACIONAL          2093       59.94
INTERNACIONAL          1399       40.06


In [40]:
print("VERIFICACIÓN DE LA NORMALIZACIÓN DEL CP\n")

verificacion_cp = con.execute("""
    SELECT 
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE codigo_postal_norm IS NULL) AS cp_norm_nulos,
        COUNT(*) FILTER (WHERE LENGTH(codigo_postal_norm) = 5 
                         AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]+$')) AS cp_norm_5_digitos,
        COUNT(*) FILTER (WHERE codigo_postal_norm IS NOT NULL 
                         AND (LENGTH(codigo_postal_norm) != 5 
                              OR NOT REGEXP_MATCHES(codigo_postal_norm, '^[0-9]+$'))) AS cp_norm_otros
    FROM silver.dim_cliente
""").fetchdf()

print(verificacion_cp.to_string(index=False))

VERIFICACIÓN DE LA NORMALIZACIÓN DEL CP

 total  cp_norm_nulos  cp_norm_5_digitos  cp_norm_otros
  3492              2               2928            562


In [41]:
print("EJEMPLOS DE CADA TIPO DE CLIENTE\n")

print("Clientes NACIONALES (5 ejemplos):")
ejemplos_es = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, 
           codigo_postal_original, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'NACIONAL'
    LIMIT 5
""").fetchdf()
print(ejemplos_es.to_string(index=False))

print("\n\nClientes INTERNACIONALES (5 ejemplos):")
ejemplos_no_es = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente,
           codigo_postal_original, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'INTERNACIONAL'
    LIMIT 5
""").fetchdf()
print(ejemplos_no_es.to_string(index=False))

EJEMPLOS DE CADA TIPO DE CLIENTE

Clientes NACIONALES (5 ejemplos):
id_cliente           nombre_cliente localidad_cliente codigo_postal_original codigo_postal_norm codigo_provincia_cliente
         1  PEREZ RODRIGUEZ, AMADOR              VIGO                  36203              36203                      036
      1307        AROHA IBERICA, SL              VIGO                  36207              36207                       36
      1324 PATRICIA GONZALEZ FERRAL              VIGO                  36214              36214                       36
        17    PEREZ GONZALEZ, JORGE              Vigo                  36305              36305                       36
      1964          SELMARK, S.L.U.              VIGO                  36315              36315                      036


Clientes INTERNACIONALES (5 ejemplos):
id_cliente             nombre_cliente localidad_cliente codigo_postal_original codigo_postal_norm codigo_provincia_cliente
      1010       PERFECT FIT LINGERIE     

In [42]:
# Investigar cuántos clientes tienen CP español pero provincia con formato no estándar
print("ANÁLISIS DE CASOS LÍMITE: CP español + provincia con formato distinto\n")

casos_limite = con.execute("""
    WITH analisis AS (
        SELECT 
            id_cliente,
            nombre_cliente,
            localidad_cliente,
            codigo_postal_original,
            codigo_postal_norm,
            codigo_provincia_cliente,
            tipo_mercado,
            -- Extraer los 2 primeros dígitos del CP normalizado (si es numérico)
            CASE 
                WHEN LENGTH(codigo_postal_norm) = 5 
                     AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]{5}$')
                THEN SUBSTR(codigo_postal_norm, 1, 2)
                ELSE NULL
            END AS provincia_segun_cp
        FROM silver.dim_cliente
    )
    SELECT 
        provincia_segun_cp,
        codigo_provincia_cliente,
        tipo_mercado,
        COUNT(*) AS num_clientes
    FROM analisis
    WHERE provincia_segun_cp IS NOT NULL
      AND TRY_CAST(provincia_segun_cp AS INTEGER) BETWEEN 1 AND 52
      AND tipo_mercado = 'INTERNACIONAL'
    GROUP BY provincia_segun_cp, codigo_provincia_cliente, tipo_mercado
    ORDER BY num_clientes DESC
    LIMIT 20
""").fetchdf()

print("Top 20 casos límite (clientes con CP español pero marcados como internacionales):\n")
print(casos_limite.to_string(index=False))

# Total de casos
total_casos = con.execute("""
    SELECT COUNT(*) AS total
    FROM silver.dim_cliente
    WHERE LENGTH(codigo_postal_norm) = 5 
      AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]{5}$')
      AND TRY_CAST(SUBSTR(codigo_postal_norm, 1, 2) AS INTEGER) BETWEEN 1 AND 52
      AND tipo_mercado = 'INTERNACIONAL'
""").fetchone()[0]

print(f"\nTotal de casos límite: {total_casos} clientes")

ANÁLISIS DE CASOS LÍMITE: CP español + provincia con formato distinto

Top 20 casos límite (clientes con CP español pero marcados como internacionales):

provincia_segun_cp codigo_provincia_cliente  tipo_mercado  num_clientes
                20                      410 INTERNACIONAL            24
                16                      410 INTERNACIONAL            24
                25                      410 INTERNACIONAL            17
                01                       NO INTERNACIONAL            15
                02                      471 INTERNACIONAL            14
                09                      471 INTERNACIONAL            13
                08                      471 INTERNACIONAL            12
                40                      410 INTERNACIONAL            12
                03                      471 INTERNACIONAL            11
                03                       NO INTERNACIONAL            10
                21                      410 INTERNACIO

In [43]:
# Investigar las LOCALIDADES de los casos límite para entender qué son realmente
print("LOCALIDADES DE LOS CASOS LÍMITE\n")

print("=== CASO 1: provincia_segun_cp=36 + provincia_cliente='036' (los del tipo PEREZ RODRIGUEZ) ===")
caso_036 = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE SUBSTR(codigo_postal_norm, 1, 2) = '36'
      AND codigo_provincia_cliente = '036'
      AND tipo_mercado = 'INTERNACIONAL'
    LIMIT 10
""").fetchdf()
print(caso_036.to_string(index=False))

print("\n\n=== CASO 2: CP español + provincia='410' (sospechosos de extranjeros) ===")
caso_410 = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE codigo_provincia_cliente = '410'
      AND tipo_mercado = 'INTERNACIONAL'
    LIMIT 10
""").fetchdf()
print(caso_410.to_string(index=False))

print("\n\n=== CASO 3: CP español + provincia='NO' (Noruega?) ===")
caso_no = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE codigo_provincia_cliente = 'NO'
      AND tipo_mercado = 'INTERNACIONAL'
    LIMIT 10
""").fetchdf()
print(caso_no.to_string(index=False))

print("\n\n=== CASO 4: CP español + provincia=NULL ===")
caso_null = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE codigo_provincia_cliente IS NULL
      AND LENGTH(codigo_postal_norm) = 5
      AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]{5}$')
      AND TRY_CAST(SUBSTR(codigo_postal_norm, 1, 2) AS INTEGER) BETWEEN 1 AND 52
    LIMIT 10
""").fetchdf()
print(caso_null.to_string(index=False))

LOCALIDADES DE LOS CASOS LÍMITE

=== CASO 1: provincia_segun_cp=36 + provincia_cliente='036' (los del tipo PEREZ RODRIGUEZ) ===
Empty DataFrame
Columns: [id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente]
Index: []


=== CASO 2: CP español + provincia='410' (sospechosos de extranjeros) ===
id_cliente                    nombre_cliente             localidad_cliente codigo_postal_norm codigo_provincia_cliente
      1470             SORTINO GIAN BATTISTA CANALICCHIO TREMESTIERI ETNEO              95030                      410
      1471                ALESSANDRO CARRARO      CARDANO AL CAMPO  VARESE              21010                      410
      1472   VANITY SNC DI OLLA GINETTA & C.                        NOVARA              28100                      410
      1474 IL BACO DA SETA DI GIUNTA DANIELA                        PESARO              61122                      410
      1476                AROSIO MARIAGRAZIA                      BIASSO

In [44]:
# Calcular cuántos clientes serían rescatados con la nueva regla
print("CLIENTES QUE SERÍAN RESCATADOS COMO NACIONALES\n")

rescatados = con.execute("""
    SELECT 
        codigo_provincia_cliente AS prov_actual,
        SUBSTR(codigo_postal_norm, 1, 2) AS prov_segun_cp,
        COUNT(*) AS num_clientes,
        COUNT(DISTINCT localidad_cliente) AS localidades_distintas
    FROM silver.dim_cliente
    WHERE LENGTH(codigo_postal_norm) = 5
      AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]{5}$')
      AND TRY_CAST(SUBSTR(codigo_postal_norm, 1, 2) AS INTEGER) BETWEEN 1 AND 52
      -- Provincia de 3 dígitos numéricos que empieza por 0
      AND LENGTH(codigo_provincia_cliente) = 3
      AND REGEXP_MATCHES(codigo_provincia_cliente, '^0[0-9]{2}$')
      -- Y los 2 últimos dígitos coinciden con los 2 primeros del CP
      AND SUBSTR(codigo_provincia_cliente, 2, 2) = SUBSTR(codigo_postal_norm, 1, 2)
      AND tipo_mercado = 'INTERNACIONAL'
    GROUP BY codigo_provincia_cliente, SUBSTR(codigo_postal_norm, 1, 2)
    ORDER BY num_clientes DESC
""").fetchdf()

print(rescatados.to_string(index=False))
print(f"\nTotal clientes rescatados: {rescatados['num_clientes'].sum()}")

CLIENTES QUE SERÍAN RESCATADOS COMO NACIONALES

Empty DataFrame
Columns: [prov_actual, prov_segun_cp, num_clientes, localidades_distintas]
Index: []

Total clientes rescatados: 0


## 5. Análisis geográfico tras la limpieza

Se verifica la distribución geográfica de los clientes nacionales por código de provincia. Esta información será útil para el análisis de geomarketing.

In [45]:
print("TOP 15 PROVINCIAS POR NÚMERO DE CLIENTES (clientes nacionales)\n")

top_provincias = con.execute("""
    SELECT 
        codigo_provincia_cliente AS cod_provincia,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'NACIONAL'
    GROUP BY codigo_provincia_cliente
    ORDER BY num_clientes DESC
    LIMIT 15
""").fetchdf()

print(top_provincias.to_string(index=False))

TOP 15 PROVINCIAS POR NÚMERO DE CLIENTES (clientes nacionales)

cod_provincia  num_clientes
          NaN           247
           08           222
           36            93
           33            91
           28            85
           46            84
           15            77
           38            57
           03            56
           43            53
           41            53
           17            50
           20            49
           48            44
           07            42


In [46]:
print("=" * 60)
print("DATOS COMPLEMENTARIOS PARA ACTUALIZAR MARKDOWNS")
print("=" * 60)

# 1. Distribución de formatos del codigo_provincia_cliente en bronze
print("\n--- 1. FORMATOS DEL codigo_provincia_cliente (bronze) ---")
formatos = con.execute("""
    SELECT 
        CASE 
            WHEN codigo_provincia_cliente IS NULL THEN 'NULL'
            WHEN TRIM(codigo_provincia_cliente) = '' THEN 'Vacío'
            WHEN REGEXP_MATCHES(codigo_provincia_cliente, '^[0-9]{2}$') THEN '2 dígitos numéricos'
            WHEN REGEXP_MATCHES(codigo_provincia_cliente, '^[0-9]{3}$') THEN '3 dígitos numéricos'
            WHEN REGEXP_MATCHES(codigo_provincia_cliente, '^[A-Za-z]{2}$') THEN '2 letras'
            WHEN REGEXP_MATCHES(codigo_provincia_cliente, '^[A-Za-z]{3}$') THEN '3 letras'
            ELSE 'Otro'
        END AS formato,
        COUNT(*) AS clientes
    FROM bronze.dim_cliente
    GROUP BY formato
    ORDER BY clientes DESC
""").fetchdf()
print(formatos.to_string(index=False))
print(f"\nTotal: {formatos['clientes'].sum()}")

# 2. Investigar clientes nacionales con codigo_provincia_cliente NULL
print("\n--- 2. CLIENTES NACIONALES CON provincia NULL ---")
nac_sin_prov = con.execute("""
    SELECT 
        COUNT(*) AS total_nacionales_sin_provincia,
        COUNT(DISTINCT SUBSTR(codigo_postal_norm, 1, 2)) AS provincias_distintas_segun_cp,
        MIN(codigo_postal_norm) AS cp_min,
        MAX(codigo_postal_norm) AS cp_max
    FROM silver.dim_cliente
    WHERE es_cliente_espanol = TRUE
      AND codigo_provincia_cliente IS NULL
""").fetchdf()
print(nac_sin_prov.to_string(index=False))

# 3. Top códigos numéricos de 3 dígitos (para validar los significados)
print("\n--- 3. TOP CÓDIGOS DE 3 DÍGITOS NUMÉRICOS (códigos de país internos de Selmark) ---")
codigos_3dig = con.execute("""
    SELECT 
        codigo_provincia_cliente AS codigo,
        COUNT(*) AS clientes,
        STRING_AGG(DISTINCT localidad_cliente, ', ') AS ejemplos_localidades
    FROM bronze.dim_cliente
    WHERE REGEXP_MATCHES(codigo_provincia_cliente, '^[0-9]{3}$')
    GROUP BY codigo_provincia_cliente
    ORDER BY clientes DESC
    LIMIT 10
""").fetchdf()
# Truncar ejemplos para que no sean demasiado largos
codigos_3dig['ejemplos_localidades'] = codigos_3dig['ejemplos_localidades'].str[:80] + '...'
print(codigos_3dig.to_string(index=False))

DATOS COMPLEMENTARIOS PARA ACTUALIZAR MARKDOWNS

--- 1. FORMATOS DEL codigo_provincia_cliente (bronze) ---
            formato  clientes
2 dígitos numéricos      1832
3 dígitos numéricos      1173
               Otro       274
           2 letras       156
           3 letras        57

Total: 3492

--- 2. CLIENTES NACIONALES CON provincia NULL ---
 total_nacionales_sin_provincia  provincias_distintas_segun_cp cp_min cp_max
                            247                             43  01400  50600

--- 3. TOP CÓDIGOS DE 3 DÍGITOS NUMÉRICOS (códigos de país internos de Selmark) ---
codigo  clientes                                                                ejemplos_localidades
   410       442 PESARO, SENIGALLIA, SARNICO, LAIGUEGLIA, RECCO, BARI, FOGGIA, CASORATE, SANT' AG...
   231       256 FELGUEIRAS, COSTA DA CAPARICA, AMARANTE - LIXA, PAÇOS DE FERREIRA, VILA NOVA DE ...
   000        84 JURATA, PRZEMYSL, Chelmsford, WEJHEROWO, KRAKOW, KOLOBRZEG, WOLOMIN, Warszawa, S...
   471

## 5.1 Análisis exhaustivo del campo `codigo_provincia_cliente`

El examen detallado del campo `codigo_provincia_cliente` sobre los 3.492 registros del maestro de clientes ha permitido identificar cinco patrones de codificación diferenciados:

| Formato | Clientes | Interpretación |
|---|---|---|
| Dos dígitos numéricos (01-52) | 1.832 | Provincias españolas en formato estándar |
| Tres dígitos numéricos | 1.173 | Códigos numéricos de país en la codificación interna del ERP de Selmark |
| Dos letras | 156 | Códigos ISO de país (PT, FR, IT y otros) |
| Tres letras | 57 | Códigos ISO de país (CAN, USA y otros) |
| Otros formatos o nulos | 274 | Registros sin patrón identificado |
| Total | 3.492 | |

### Identificación de los códigos numéricos de tres dígitos

El subconjunto de 1.173 clientes con código de provincia de tres dígitos numéricos corresponde a un esquema de codificación interna del ERP cuyo significado no se documenta en los metadatos disponibles. Para inferir la equivalencia entre código y país se ha cruzado el código de provincia con la localidad declarada por el cliente, obteniendo los siguientes resultados para los códigos más representativos:

| Código | Casos | País o región | Localidades de referencia |
|---|---|---|---|
| `410` | 442 | Italia (centro-norte) | Pesaro, Senigallia, Bari, Foggia, Recco |
| `231` | 256 | Portugal | Felgueiras, Amarante, Paços de Ferreira |
| `000` | 84 | Polonia | Kraków, Warszawa, Kolobrzeg, Wejherowo |
| `471` | 59 | Bélgica | Gent, Brugge, Hamont, Wetteren |
| `299` | 26 | Portugal (región del Minho) | Melgaço, Monção |
| `036` | 24 | España (Pontevedra) | Vigo, Oia, A Guarda |
| `385` | 14 | Italia (Lazio) | Roma, Ostia |
| `360` | 12 | Italia (Campania) | Capri, Acerra, Napoli |
| `216` | 12 | Portugal (región del Minho) | Gandra, Arcos de Valdevez, Valença |
| `213` | 12 | Portugal (norte) | Porto, Vila Nova de Gaia, Maia |

### Identificación de un patrón de error en el ERP

El análisis pormenorizado del subconjunto de tres dígitos ha permitido detectar un grupo reducido de registros con código de provincia de patrón `0XX`, donde `XX` corresponde al código de una provincia española. La coherencia entre el código postal asociado y la localidad declarada del cliente confirma que se trata de un error de codificación en el ERP de origen, en el cual el código provincial de dos dígitos se ha almacenado con un cero a la izquierda. Los casos identificados se concentran en el código `036`, correspondiente a la provincia de Pontevedra (24 clientes), grupo en el que se incluye la propia sede de la compañía.

Como referencia, los registros más representativos son los siguientes:

- `id_cliente=1`: Pérez Rodríguez, Amador — Vigo — CP 36203 — provincia `036`
- `id_cliente=1964`: Selmark, S.L.U. — Vigo — CP 36315 — provincia `036`

### Lógica final del flag `es_cliente_espanol`

A partir de los hallazgos anteriores, la definición del flag `es_cliente_espanol` combina dos reglas mediante un operador de unión lógica. La primera, aplicable al formato estándar, exige que el código de provincia sea numérico de dos dígitos en el rango 01-52 y que coincida con los dos primeros dígitos del código postal normalizado. La segunda, aplicable al formato alternativo identificado, exige que el código de provincia sea numérico de tres dígitos con patrón `0XX` y que los dos últimos dígitos coincidan con los dos primeros del código postal normalizado. Ambas reglas requieren adicionalmente que el código postal sea numérico de hasta cinco dígitos.

Esta validación cruzada resulta necesaria para evitar falsos positivos en la clasificación, dado que países como Italia, Noruega o Polonia disponen de códigos postales numéricos de cinco dígitos que, tras la normalización mediante padding, podrían interpretarse erróneamente como códigos postales españoles si no se contrastasen con el campo de provincia.

De forma complementaria, el análisis identifica un grupo de 247 clientes españoles cuyo `codigo_provincia_cliente` no está informado en el origen pero cuyo código postal pertenece inequívocamente al rango español (01400 a 50600, cubriendo 43 provincias distintas). Estos registros quedan correctamente clasificados como nacionales mediante la coherencia del código postal, sin requerir información adicional sobre la provincia. Su inclusión resulta metodológicamente relevante, ya que su exclusión supondría la pérdida de un volumen no despreciable de clientes en los análisis geográficos nacionales posteriores.

## 5.2 Decisión sobre la tipificación de clientes (a desarrollar en Gold)

Tras la conversación con el tutor, se confirma que la diferenciación nacional/internacional **no es suficiente** para abordar correctamente los análisis posteriores de clustering y machine learning. Selmark opera con una cartera que combina canales muy heterogéneos:

| Canal | Naturaleza | Comportamiento esperado |
|---|---|---|
| B2B nacional | Tiendas multimarca, distribuidores | Pedidos voluminosos, regularidad, ticket alto |
| El Corte Inglés (ECI) | Operativa de venta a gran cuenta | Pedidos masivos, devoluciones gestionadas |
| Ecommerce | Compras de particulares vía web | Tickets pequeños, alta frecuencia, sin previsibilidad |
| Depósito | Operativa logística específica | Movimientos de stock, no venta directa |
| Muestras | Comerciales y promocionales | Volumen bajo, sin facturación directa |
| Distribución internacional | Partners y exportación | Volúmenes muy variables, calendarios distintos |

### Implicación para los modelos

Mezclar todos los canales en un mismo clustering produciría grupos dominados por las diferencias de canal en lugar de por el comportamiento real del cliente dentro de su canal. Por ejemplo, un buen cliente de Ecommerce y un buen cliente B2B son comparables solo dentro de su segmento, no entre segmentos.

### Estrategia de implementación

La tipología de cliente (`tipo_cliente`) **no se construirá en Silver**, ya que requiere agregar las operaciones de cada cliente desde las tablas de hechos. Se calculará en la capa Gold (`gold.cliente_360`) a partir del mix de tipos de operación realizadas, asignando a cada cliente su perfil dominante. Posteriormente, los análisis de clustering y machine learning se ejecutarán dentro de cada tipo y no sobre el conjunto completo de la cartera.

Esta decisión metodológica garantiza que los modelos posteriores capturen patrones reales de comportamiento dentro de poblaciones homogéneas, en lugar de diferencias estructurales entre canales.

## 6. Conclusiones y siguientes pasos

### Resultado obtenido

La capa Silver del maestro de clientes se materializa en la tabla `silver.dim_cliente`, que contiene 3.492 registros equivalentes al volumen íntegro de la capa Bronze. Sobre esta tabla se han aplicado tres transformaciones principales: la normalización del código postal a cinco dígitos para los clientes españoles, conservándose el formato original en una columna paralela con fines de trazabilidad; la creación del flag booleano `es_cliente_espanol` y de la categoría textual `tipo_mercado` con valores `NACIONAL` e `INTERNACIONAL`; y la limpieza sistemática de espacios en blanco en los campos de texto.

### Composición de la cartera

La aplicación de las reglas de clasificación geográfica sobre el maestro de clientes arroja la siguiente distribución:

| Tipo de mercado | Clientes | Porcentaje |
|---|---|---|
| NACIONAL | 2.093 | 59,94 % |
| INTERNACIONAL | 1.399 | 40,06 % |
| Total | 3.492 | 100,00 % |

La comparación con la versión inicial del maestro (3.469 registros, con un 53,2 % nacional y un 46,8 % internacional) revela un desplazamiento significativo de la composición hacia el mercado nacional. Este cambio tiene su origen en una actualización del sistema ERP corporativo en la que se corrigieron errores de codificación en el campo `codigo_provincia_cliente` de un conjunto relevante de registros que previamente figuraban catalogados como internacionales y que, tras la revisión, han pasado a clasificarse correctamente como clientes nacionales.

### Estrategia de uso de la cartera en el proyecto

La cartera de Selmark presenta dos dimensiones de heterogeneidad que deben tratarse de forma diferenciada en los análisis posteriores.

La primera dimensión, de naturaleza geográfica y materializada en el campo `tipo_mercado`, distingue entre los 2.093 clientes nacionales, que constituyen la población objetivo del análisis técnico del proyecto (clustering, geomarketing y modelado predictivo), y los 1.399 clientes internacionales, mayoritariamente vinculados al canal de comercio electrónico y a la operativa de exportación. Estos últimos quedarán reservados para la sección de análisis estratégico de negocio, donde aportan información agregada sobre la dimensión exterior de la compañía.

La segunda dimensión, de naturaleza operativa, se construirá en la capa Gold a partir del mix de operaciones realizadas por cada cliente y se materializará en el campo `tipo_cliente`. Esta tipología permitirá segmentar la cartera nacional en perfiles homogéneos (B2B, El Corte Inglés, comercio electrónico nacional, depósito y otros) sobre los cuales aplicar técnicas de clustering y modelos de aprendizaje automático con la garantía de homogeneidad muestral que dichas técnicas requieren.

Los análisis posteriores combinarán ambas dimensiones para asegurar que los modelos se entrenan sobre poblaciones internamente comparables.

### Validaciones superadas

- Conservación íntegra del volumen entre origen y destino, sin pérdidas ni duplicaciones.
- Unicidad de la clave primaria `id_cliente`.
- Coherencia de los tipos de datos finales (VARCHAR para identificadores, BOOLEAN para flags).
- Separación correcta entre provincias españolas estándar y códigos numéricos de país asignados al esquema interno del ERP.

### Implicaciones para los siguientes notebooks

La tabla `silver.dim_cliente` constituye la dimensión de referencia para todas las uniones que se realicen sobre el campo `id_cliente` en los notebooks posteriores. El campo `codigo_postal_norm` actuará como clave de unión con la tabla `silver.mosaic` para el enriquecimiento sociodemográfico, y el filtro `tipo_mercado = 'NACIONAL'` se aplicará por defecto en los análisis técnicos del proyecto.

### Próximo notebook

`04_silver_fact_lineas_pedido.ipynb` — Construcción de la tabla de hechos de líneas de pedido depurada, con aplicación de la ventana temporal 2022-2025, exclusión de líneas anuladas y cálculo del intervalo temporal transcurrido entre la fecha de pedido y la fecha de entrega.

In [47]:
# ============================================================
# CIERRE DE LA SESIÓN
# ============================================================
# Libera la conexión a DuckDB para evitar bloqueos en otros notebooks

try:
    con.close()
    print("Conexión a DuckDB cerrada correctamente.")
except Exception as e:
    print(f"Aviso al cerrar conexión: {e}")

import gc
gc.collect()

print("\nTabla silver.dim_cliente persistida en disco.")
print("La base está libre para otros notebooks.")
print("Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.")

Conexión a DuckDB cerrada correctamente.

Tabla silver.dim_cliente persistida en disco.
La base está libre para otros notebooks.
Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.
